# Full preprocessing pipeline test

Runs the complete pipeline on one image and visualises every stage:
1. Raw → 2. Gaussian bg subtraction → 3. MIP → 4. Percentile norm → 5. Patches

In [ ]:
import sys, os, gc, tempfile
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', '..')))

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from utils_data.preprocess_pseudolabels import (
    load_image, denoise_volume, maximum_intensity_projection,
    normalize_percentile, extract_patches, process_single_image,
)

In [ ]:
# --- Config ---
IMAGE_PATH = Path(
    r"C:\Users\t-vanokhina\repos\thesis\data\Microscopy\20251030"
    r"\_2_3_PSYHARMIN_7_Multichannel Z-Stack_20251030_72_"
    r"\stack1\frame_t_0.ets"
)
ROLLING_BALL_RADIUS = 50
MEDIAN_SIZE = 0          # 0 = off
PLOW, PHIGH = 1.0, 99.8
PATCH_SIZE = 128


## 1. Load raw volume

In [ ]:
volume = load_image(IMAGE_PATH)  # (C, Z, Y, X)
C, Z, H, W = volume.shape
print(f"Shape: C={C}, Z={Z}, H={H}, W={W}, dtype={volume.dtype}")

In [ ]:
# Show middle Z-slice per channel (raw)
mid_z = Z // 2
fig, axes = plt.subplots(1, C, figsize=(5 * C, 5))
if C == 1: axes = [axes]
for ch in range(C):
    axes[ch].imshow(volume[ch, mid_z], cmap='gray')
    axes[ch].set_title(f'Raw  ch{ch}  z={mid_z}')
    axes[ch].axis('off')
plt.suptitle('Step 1: Raw (middle Z-slice)', fontsize=10)
plt.tight_layout(); plt.show()

## 2. Denoise (Gaussian bg subtraction, per Z-slice)

In [ ]:
volume_clean = denoise_volume(volume, median_size=MEDIAN_SIZE,
                              rolling_ball_radius=ROLLING_BALL_RADIUS)
print(f"Denoised shape: {volume_clean.shape}")

In [ ]:
# Compare raw vs denoised on the same Z-slice
fig, axes = plt.subplots(2, C, figsize=(5 * C, 8))
if C == 1: axes = axes.reshape(-1, 1)
for ch in range(C):
    vmax = np.percentile(volume[ch, mid_z], 99)
    axes[0, ch].imshow(volume[ch, mid_z],
                       cmap='gray', vmin=0, vmax=vmax)
    axes[0, ch].set_title(f'Raw  ch{ch}', fontsize=9)
    axes[0, ch].axis('off')
    axes[1, ch].imshow(volume_clean[ch, mid_z],
                       cmap='gray', vmin=0, vmax=vmax)
    axes[1, ch].set_title(f'Denoised  ch{ch}', fontsize=9)
    axes[1, ch].axis('off')
plt.suptitle('Step 2: Raw vs Denoised (same Z-slice, same scale)', fontsize=10)
plt.tight_layout(); plt.show()

## 3. MIP (old vs new)

In [ ]:
mip_old = maximum_intensity_projection(volume)
mip_new = maximum_intensity_projection(volume_clean)
del volume, volume_clean; gc.collect()
print(f"MIP shape: {mip_old.shape}")

In [ ]:
fig, axes = plt.subplots(2, C, figsize=(5 * C, 8))
if C == 1: axes = axes.reshape(-1, 1)
for ch in range(C):
    vmax = np.percentile(mip_old[ch], 99)
    axes[0, ch].imshow(mip_old[ch],
                       cmap='gray', vmin=0, vmax=vmax)
    axes[0, ch].set_title(f'MIP old  ch{ch}', fontsize=9)
    axes[0, ch].axis('off')
    axes[1, ch].imshow(mip_new[ch],
                       cmap='gray', vmin=0, vmax=vmax)
    axes[1, ch].set_title(f'MIP new  ch{ch}', fontsize=9)
    axes[1, ch].axis('off')
plt.suptitle('Step 3: MIP — old (no denoise) vs new (Gaussian bg sub)', fontsize=10)
plt.tight_layout(); plt.show()

## 4. Percentile normalization

In [ ]:
norm_old = normalize_percentile(mip_old, plow=PLOW, phigh=PHIGH)
norm_new = normalize_percentile(mip_new, plow=PLOW, phigh=PHIGH)
del mip_old, mip_new; gc.collect()

In [ ]:
fig, axes = plt.subplots(2, C, figsize=(5 * C, 8))
if C == 1: axes = axes.reshape(-1, 1)
for ch in range(C):
    axes[0, ch].imshow(norm_old[ch],
                       cmap='gray', vmin=0, vmax=1)
    axes[0, ch].set_title(f'Norm old  ch{ch}', fontsize=9)
    axes[0, ch].axis('off')
    axes[1, ch].imshow(norm_new[ch],
                       cmap='gray', vmin=0, vmax=1)
    axes[1, ch].set_title(f'Norm new  ch{ch}', fontsize=9)
    axes[1, ch].axis('off')
plt.suptitle('Step 4: After percentile norm [0,1] — old vs new', fontsize=10)
plt.tight_layout(); plt.show()

for ch in range(C):
    print(f"ch{ch}  old: mean={norm_old[ch].mean():.4f}  std={norm_old[ch].std():.4f}"
          f"  |  new: mean={norm_new[ch].mean():.4f}  std={norm_new[ch].std():.4f}")

## 5. Patches (sample grid)

In [ ]:
patches_new = extract_patches(norm_new, patch_size=PATCH_SIZE)
print(f"Patches: {patches_new.shape}  "
      f"({H // PATCH_SIZE}x{W // PATCH_SIZE} grid)")

# Show 4x4 random patches from channel 0
rng = np.random.default_rng(42)
idxs = rng.choice(patches_new.shape[0], size=16, replace=False)
fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for ax, i in zip(axes.flat, idxs):
    ax.imshow(patches_new[i, 0], cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'#{i}', fontsize=7)
    ax.axis('off')
plt.suptitle(f'Step 5: 16 random patches (ch0, {PATCH_SIZE}×{PATCH_SIZE})', fontsize=10)
plt.tight_layout(); plt.show()

## 6. Full `process_single_image` end-to-end

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    records = process_single_image(
        path=IMAGE_PATH,
        output_dir=Path(tmp),
        patch_size=PATCH_SIZE,
        plow=PLOW, phigh=PHIGH,
        image_index=0,
        median_size=MEDIAN_SIZE,
        rolling_ball_radius=ROLLING_BALL_RADIUS,
    )
    # Load back a few saved patches to verify
    saved = [np.load(Path(tmp) / r['filename']) for r in records[:4]]

print(f"process_single_image → {len(records)} patches saved")
print(f"Sample record keys: {list(records[0].keys())}")

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, p, rec in zip(axes, saved, records):
    ax.imshow(p[0], cmap='gray', vmin=0, vmax=1)
    ax.set_title(rec['filename'], fontsize=7)
    ax.axis('off')
plt.suptitle('Patches reloaded from disk (.npy)', fontsize=10)
plt.tight_layout(); plt.show()